# RL-RAG Checkpoint 1 - Baseline Advanced RAG + GOLD-5 Eval

**Run All** top-to-bottom.

Notebook flow:
1) Fix Windows OpenMP conflicts (prevents kernel crash)
2) Find repo root + set paths
3) Build baseline advanced RAG (rewrite → hybrid retrieve → rerank → answer)
4) 3 sanity queries (prints top-k chunks + excerpts)
5) Baseline evaluation on `data/evalset_5_gold.jsonl` → saves to `results/`
6) RL env demo (3 actions on 1 query)


In [1]:
import os
os.environ["OLLAMA_BASE_URL"] = "http://127.0.0.1:11434"   # force IPv4
print("OLLAMA_BASE_URL =", os.environ["OLLAMA_BASE_URL"])

OLLAMA_BASE_URL = http://127.0.0.1:11434


In [2]:
# -------------------------------
# MUST RUN FIRST (Windows fix)
# Prevent OpenMP duplicate runtime crash (faiss/torch issue on Windows)
# -------------------------------
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("OpenMP env set.")

OpenMP env set.


In [3]:
import sys
from pathlib import Path

# Robust repo-root detection: walk up until we find 'rag/' folder
cwd = Path.cwd().resolve()
p = cwd
while p != p.parent and not (p / "rag").exists():
    p = p.parent

if not (p / "rag").exists():
    raise RuntimeError(f"Could not find repo root containing 'rag/' starting from {cwd}")

REPO_ROOT = p
sys.path.insert(0, str(REPO_ROOT))

PDF_PATH = REPO_ROOT / "data" / "prudentservices_dataset_rag.pdf"
EVALSET_PATH = REPO_ROOT / "data" / "evalset_5_gold.jsonl"
RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("CWD:", cwd)
print("REPO_ROOT:", REPO_ROOT)
print("rag exists:", (REPO_ROOT / "rag").exists())
print("PDF exists:", PDF_PATH.exists(), PDF_PATH)
print("Evalset exists:", EVALSET_PATH.exists(), EVALSET_PATH)
print("RESULTS_DIR:", RESULTS_DIR)

CWD: C:\Users\Admin\Desktop\RL-RAG\notebooks
REPO_ROOT: C:\Users\Admin\Desktop\RL-RAG
rag exists: True
PDF exists: True C:\Users\Admin\Desktop\RL-RAG\data\prudentservices_dataset_rag.pdf
Evalset exists: True C:\Users\Admin\Desktop\RL-RAG\data\evalset_5_gold.jsonl
RESULTS_DIR: C:\Users\Admin\Desktop\RL-RAG\results


In [4]:
# Core RAG
from rag.pdf_ingest import load_or_build_chunks
from rag.index_bm25 import BM25Index
from rag.index_dense import DenseIndex
from rag.rewrite import OllamaRewriter
from rag.rerank import CrossEncoderReranker
from rag.llm import OllamaLLM
from rag.pipeline import RAGPipeline
from rag.config import RAGConfig

# Eval
from rag.evalset import load_evalset_jsonl
from rag.evaluate import evaluate_items
from rag.reporting import save_metrics_summary_json

# Env demo
from env.rag_env import RAGEnv

# Quick check: ensure correct metrics module is loaded
import rag.metrics as _m
print("rag.metrics file:", _m.__file__)
print("Has MetricsRow:", hasattr(_m, "MetricsRow"))

rag.metrics file: C:\Users\Admin\Desktop\RL-RAG\rag\metrics.py
Has MetricsRow: True


In [5]:
# Chunking + caching (cache stored at data/cache)
chunks, cache_path = load_or_build_chunks(
    str(PDF_PATH),
    chunk_size=450,
    overlap=80,
    cache_dir=str(REPO_ROOT / "data" / "cache"),
    force_rebuild=False,
)
print("Num chunks:", len(chunks))
print("Chunk cache:", cache_path)
print("Sample chunk:", chunks[0].chunk_id, chunks[0].meta)
print(chunks[0].text[:220])

Num chunks: 125
Chunk cache: C:\Users\Admin\Desktop\RL-RAG\data\cache\chunks_dd5685b4b4a769b1.jsonl
Sample chunk: c000000 {'source': 'prudentservices_dataset_rag.pdf', 'page': 1, 'char_start': 0, 'char_end': 450}
Prudent Services Dataset Pack 
Public Website Reference + Internal Document Templates 
Disclosure 
Part I is a structured reference that paraphrases publicly available information from 
prudentservices.co.in. 
 
Part II 


In [6]:
# Build BM25
bm25 = BM25Index()
bm25.build(chunks)
print("BM25 ready.")

# Build Dense index with Ollama embeddings (cache stored at data/cache)
dense = DenseIndex(embed_model="nomic-embed-text")
dense.build(
    chunks,
    batch_size=32,
    cache_dir=str(REPO_ROOT / "data" / "cache"),
    use_cache=True,
)
print("Dense index ready (Ollama embeddings).")

BM25 ready.
Dense index ready (Ollama embeddings).


In [7]:
# Initialize models
rewriter = OllamaRewriter()              # retrieval-only rewrite + multi-query
reranker = CrossEncoderReranker()        # cross-encoder rerank
llm = OllamaLLM()                        # final answer generation

pipe = RAGPipeline(
    bm25=bm25,
    dense=dense,
    rewriter=rewriter,
    reranker=reranker,
    llm=llm,
)

# Baseline config (advanced features ON)
cfg = RAGConfig(
    rewrite=True,
    nq=3,
    bm25_kcand=50,
    dense_kcand=50,
    x_bm25=0.5,
    y_dense=0.5,
    rerank=True,
    rerank_topR=30,
    k_final=10,
)

print("Baseline config:", cfg.to_dict())

Baseline config: {'rewrite': True, 'nq': 3, 'bm25_kcand': 50, 'dense_kcand': 50, 'x_bm25': 0.5, 'y_dense': 0.5, 'rerank': True, 'rerank_topR': 30, 'k_final': 10, 'memory_max_chars': 6000, 'allow_retrieve_zero': False}


In [8]:
def print_top_chunks(res, max_chars=220):
    print("\nTop-k context chunks:")
    for i, c in enumerate(res.context_chunks, start=1):
        page = c.meta.get("page", None)
        prefix = f"{i}. {c.chunk_id}"
        if page is not None:
            prefix += f" (page {page})"
        print(prefix)
        print(c.text[:max_chars].replace("\n", " "))
        print("-" * 80)

def run_sanity(session_id, raw_query, chat_history):
    res = pipe.run_rag(session_id=session_id, raw_query=raw_query, chat_history=chat_history, config=cfg)
    print("\n" + "="*110)
    print("RAW QUERY:", raw_query)
    print("RETRIEVAL QUERY:", res.retrieval_query)
    print("RETRIEVAL QUERIES:", res.retrieval_queries)
    print("-"*110)
    print("ANSWER:\n", res.answer)
    print("-"*110)
    print("TIMINGS:", res.timings)
    print("TOKENS:", res.tokens)
    print_top_chunks(res)
    return res

In [9]:
# Sanity Query 1 (direct in-domain)
chat_history = [("user", "Hi"), ("assistant", "Hello.")]
_ = run_sanity(
    session_id="sanity_1",
    raw_query="List all training topics mentioned in the public listing.",
    chat_history=chat_history,
)

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(



RAW QUERY: List all training topics mentioned in the public listing.
RETRIEVAL QUERY: List all training topics mentioned in public listings related to data science and machine learning.
RETRIEVAL QUERIES: ['```', '[', '"data science topics",']
--------------------------------------------------------------------------------------------------------------
ANSWER:
 The training topics mentioned in the public listing are:

1. Fire prevention and fire-fighting basics
2. Emergency evacuation procedures
3. Traffic control
4. Basic security procedures and SOP-based operations
5. Incident Scenarios (Template Set)
   - Gate bypass attempt
   - Medical emergency
   - Fire alarm trigger
   - Lost person report
   - Vehicle seal mismatch
   - Crowd surge at entry
   - Suspicious package
   - Power outage affecting access control

Note that some of these topics are further categorized under "Scenario Topics (Examples)" and have a common response pattern template associated with them.
---------------

In [10]:
# Sanity Query 2 (memory-dependent, tests rewrite using history)
chat_history = [
    ("user", "We are rating risks on a new site onboarding form."),
    ("assistant", "Use the risk assessment matrix scoring."),
    ("user", "What is the formula again?"),
]
_ = run_sanity(
    session_id="sanity_2",
    raw_query="What is the formula again?",
    chat_history=chat_history,
)


RAW QUERY: What is the formula again?
RETRIEVAL QUERY: risk assessment matrix scoring formula
RETRIEVAL QUERIES: ['```', '[', '"risk assessment matrix scoring formulas and templates",']
--------------------------------------------------------------------------------------------------------------
ANSWER:
 I'm ready to assist. What's the question?
--------------------------------------------------------------------------------------------------------------
TIMINGS: TimingInfo(total_s=227.15973170000052, rewrite_s=18.79846069999985, retrieve_s=0.6718614000001253, hybrid_s=0.0004906999993181671, rerank_s=0.6012118999997256, build_prompt_s=5.890000011277152e-05, gen_s=191.75300180000067)
TOKENS: TokenUsage(prompt_tokens=1121, completion_tokens=12, total_tokens=1133, context_tokens=463, memory_tokens=24)

Top-k context chunks:
1. c000013 (page 5)
ints, and visitor flow.  2.2.2 Risk Assessment Matrix  Risk score = Likelihood (1 –5) × Impact (1 –5). Use the score to decide guard posts, patrol

In [11]:
# Sanity Query 3 (memory-dependent)
chat_history = [
    ("user", "We maintain a log for keys issued and returned."),
    ("assistant", "That’s the Key Control Log."),
    ("user", "What should I do if keys are missing?"),
]
_ = run_sanity(
    session_id="sanity_3",
    raw_query="What should I do if keys are missing?",
    chat_history=chat_history,
)


RAW QUERY: What should I do if keys are missing?
RETRIEVAL QUERY: What should I do if keys are missing from the Key Control Log?
RETRIEVAL QUERIES: ['Here are three diverse search queries for retrieving documents:', '```', '["key control log missing keys", "missing key log entries", "key log recovery process"]']
--------------------------------------------------------------------------------------------------------------
ANSWER:
 If keys are missing, the incident should be escalated immediately. This is because missing keys are considered incidents that require attention and action to prevent unauthorized access or potential security breaches.

Please document the time-stamped actions taken in the occurrence book and submit an incident report form as per the Common Response Pattern (Template) [c000054]. Also, notify your supervisor and emergency services if necessary.
--------------------------------------------------------------------------------------------------------------
TIMINGS

In [12]:
# Load the gold 5-query evalset
items = load_evalset_jsonl(str(EVALSET_PATH))
print("Total eval items:", len(items))
print("In-domain:", sum(1 for x in items if x.domain == "in_domain"))
print("Out-of-domain:", sum(1 for x in items if x.domain == "out_of_domain"))

print("\nSample items:")
for it in items[:5]:
    print(it.qid, it.domain, it.raw_query[:80])

Total eval items: 5
In-domain: 3
Out-of-domain: 2

Sample items:
in_010 in_domain Name two training topics related to emergencies.
in_065 in_domain What is the formula again?
in_062 in_domain What should I do if keys are missing?
ood_001 out_of_domain What is the capital of India?
ood_002 out_of_domain Solve: 37 * 19


In [13]:
# # RL Env demo: same query, try 3 different actions and compare reward/cost
# env = RAGEnv(pipe=pipe, base_config=cfg)

# sample_item = next(x for x in items if x.domain == "in_domain")
# state = env.reset(sample_item)
# print("State vector len:", len(state))
# print("Action space size:", env.action_space_n())

# def find_action_idx(target):
#     for i, a in enumerate(env.actions):
#         if (a.rewrite, a.nq, a.rerank, a.k_final, round(a.x_bm25, 1)) == target:
#             return i
#     return None

# a_cheap = find_action_idx((0, 1, 0, 5, 0.5))   # no rewrite, no rerank, small k
# a_mid   = find_action_idx((1, 3, 0, 10, 0.5))  # rewrite+multi, no rerank, mid k
# a_full  = find_action_idx((1, 3, 1, 20, 0.5))  # rewrite+multi, rerank, big k

# action_ids = [a for a in [a_cheap, a_mid, a_full] if a is not None]
# print("Action indices:", action_ids)

# for ai in action_ids:
#     _, reward, done, info = env.step(ai)
#     m = info["metrics"]
#     print("\n--- Action", ai, info["action"], "---")
#     print("reward:", round(reward, 4), "done:", done)
#     print("correctness:", m.get("correctness"), "faithfulness:", m.get("faithfulness"), "recall@k:", m.get("evidence_recall_at_k"))
#     print("time_s:", round(m.get("total_time_s", 0.0), 3), "total_tokens:", m.get("total_tokens"), "k_final:", m.get("k_final"))

In [14]:
import json, csv, time, traceback
from dataclasses import asdict

from rag.judge import OllamaJudge
from rag.metrics import score_one
from rag.reporting import save_metrics_summary_json

run_name = "baseline_5"
details_path = RESULTS_DIR / f"{run_name}_details.jsonl"
metrics_path = RESULTS_DIR / f"{run_name}_metrics.csv"
summary_path = RESULTS_DIR / f"{run_name}_metrics.json"

judge = OllamaJudge()

rows = []
n = len(items)

print(f"Running evaluation: {run_name} on {n} queries")
print("Saving details ->", details_path)
print("Saving metrics  ->", metrics_path)

with open(details_path, "w", encoding="utf-8") as f_details:
    for i, it in enumerate(items, start=1):
        t0 = time.perf_counter()

        try:
            # Run baseline RAG
            res = pipe.run_rag(
                session_id=it.qid,
                raw_query=it.raw_query,
                chat_history=it.chat_history,
                config=cfg,
            )

            # Score
            row = score_one(it, res, judge=judge)
            rows.append(row)

            detail_obj = {
                "qid": it.qid,
                "domain": it.domain,
                "raw_query": it.raw_query,
                "gold_answer": it.gold_answer,
                "gold_support_chunk_ids": it.gold_support_chunk_ids,
                "retrieval_query": res.retrieval_query,
                "retrieval_queries": res.retrieval_queries,
                "answer": res.answer,
                "context_chunk_ids": [c.chunk_id for c in res.context_chunks],
                "context_pages": [c.meta.get("page") for c in res.context_chunks],
                "timings": asdict(res.timings),
                "tokens": asdict(res.tokens),
                "config": res.meta.get("config", {}),
                "metrics_row": asdict(row),
                "error": None,
            }
            f_details.write(json.dumps(detail_obj, ensure_ascii=False) + "\n")
            f_details.flush()

            dt = time.perf_counter() - t0
            print(
                f"[{i:02d}/{n:02d}] {it.qid} ({it.domain}) | "
                f"time={dt:6.1f}s | corr={row.correctness} faith={row.faithfulness} "
                f"tok={row.total_tokens} k={row.k_final} rerank={row.rerank_enabled}",
                flush=True
            )

        except Exception as e:
            dt = time.perf_counter() - t0
            err_msg = f"{type(e).__name__}: {e}"

            detail_obj = {
                "qid": it.qid,
                "domain": it.domain,
                "raw_query": it.raw_query,
                "gold_answer": it.gold_answer,
                "gold_support_chunk_ids": it.gold_support_chunk_ids,
                "retrieval_query": None,
                "retrieval_queries": None,
                "answer": None,
                "context_chunk_ids": [],
                "context_pages": [],
                "timings": {},
                "tokens": {},
                "config": cfg.to_dict() if hasattr(cfg, "to_dict") else {},
                "metrics_row": None,
                "error": err_msg,
            }
            f_details.write(json.dumps(detail_obj, ensure_ascii=False) + "\n")
            f_details.flush()

            print(
                f"[{i:02d}/{n:02d}] {it.qid} ({it.domain}) | FAILED after {dt:6.1f}s | {err_msg}",
                flush=True
            )
            continue

# Save CSV only if at least one row succeeded
if rows:
    with open(metrics_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(asdict(rows[0]).keys()))
        writer.writeheader()
        for r in rows:
            writer.writerow(asdict(r))

    save_metrics_summary_json(str(metrics_path), str(summary_path))

    print("\nDONE ✅ Files written:")
    print(metrics_path)
    print(details_path)
    print(summary_path)
else:
    print("\nNo successful rows were produced.")
    print("Details file still written:", details_path)

Running evaluation: baseline_5 on 5 queries
Saving details -> C:\Users\Admin\Desktop\RL-RAG\results\baseline_5_details.jsonl
Saving metrics  -> C:\Users\Admin\Desktop\RL-RAG\results\baseline_5_metrics.csv
[01/05] in_010 (in_domain) | time= 411.6s | corr=1 faith=1 tok=1090 k=10 rerank=1
[02/05] in_065 (in_domain) | time= 417.2s | corr=2 faith=1 tok=1216 k=10 rerank=1
[03/05] in_062 (in_domain) | time= 338.6s | corr=2 faith=1 tok=992 k=10 rerank=1
[04/05] ood_001 (out_of_domain) | time= 431.1s | corr=2 faith=1 tok=1261 k=10 rerank=1
[05/05] ood_002 (out_of_domain) | time= 493.1s | corr=2 faith=1 tok=1319 k=10 rerank=1
Saved summary JSON: C:\Users\Admin\Desktop\RL-RAG\results\baseline_5_metrics.json

DONE ✅ Files written:
C:\Users\Admin\Desktop\RL-RAG\results\baseline_5_metrics.csv
C:\Users\Admin\Desktop\RL-RAG\results\baseline_5_details.jsonl
C:\Users\Admin\Desktop\RL-RAG\results\baseline_5_metrics.json
